# コラム: エネルギー期待値の測定法とその統計的ゆらぎ

VQEではハミルトニアンのエネルギー期待値 $E(\boldsymbol{\theta}) = \ev{H}{\psi(\boldsymbol{\theta})}$ を用いるが、この期待値を量子コンピュータで直接測定するのは簡単ではない。
そこでNISQデバイスなどでVQEを実行する際は、式 (5.4) のようにハミルトニアンを $n$ 量子ビットパウリ演算子の和に分解し、それぞれのパウリ演算子の期待値を量子コンピュータで測定する。
このコラムでは、パウリ演算子の期待値の測定法と、その推定結果の統計的ゆらぎについて紹介する。

結論からいうと、$n$ 量子ビットのパウリ演算子 $P$ の期待値 $\ev{P}{\psi}$ は、$\ket{\psi}$ に 1 量子ビットゲートを最大 $n$ 個追加するだけで測定できる。
この事実は、(1) $P$ は最大 $n$ 個の 1 量子ビットゲートを用いることでパウリ演算子 $Z$ のみからなるパウリ演算子に「変換」できること、(2) パウリ演算子 $Z$ のみからなるパウリ演算子の期待値は第1章で学んだ量子ビットの測定結果を用いて計算できること、という 2 点から示すことができる。

まず (1) については、アダマールゲート $H$ と位相ゲート $S$ に関する等式 $HXH^{\dag}=Z$, $HS^{\dag}Y(HS^{\dag})^{\dag}=Z$ を使えばよい。
具体的には、$n$ 量子ビットパウリ演算子を $P = P^{(0)} \otimes \cdots \otimes P^{(n-1)}$（$P^{(i)}$ は $i$ 番目の量子ビットに作用する 1 量子ビットパウリ演算子 $I, X, Y, Z$ のどれか）と書いたとき、$P^{(i)} = X$ である量子ビット $i$ には $H$ を、$P^{(i)} = Y$ である量子ビット $i$ には $HS^{\dag}$ をかける量子回路を $U$ とする。
$U$ は最大 $n$ 個の 1 量子ビットゲートからなり、$UPU^{\dag}$ は恒等演算子 $I$ とパウリ演算子 $Z$ のみのテンソル積となる。

$$
\ev{P}{\psi} = \mel{\psi}{(U^{\dag}U)P(U^{\dag}U)}{\psi} = \mel{\psi}{U^{\dag}(UPU^{\dag})U}{\psi}
$$

が成り立つから、$U$ を $\ket{\psi}$ に作用させた状態 $\ket{\psi'} = U\ket{\psi}$ に対して $UPU^{\dag}$ の期待値を求めれば、もともと知りたかった期待値 $\ev{P}{\psi}$ が計算できる。

(2) に関しては、$Z = \ketbra{0} - \ketbra{1}$ であることに注意すると、例えば左端の量子ビットに作用するパウリ演算子 $Z^{(0)}$ に対して

$$
Z^{(0)} = \ketbra{0} \otimes I \otimes \cdots \otimes I - \ketbra{1} \otimes I \otimes \cdots \otimes I
$$

が成り立つ。よって

$$
\ev{Z^{(0)}}{\psi'} = \ev{(\ketbra{0} \otimes I \otimes \cdots \otimes I)}{\psi'} - \ev{(\ketbra{1} \otimes I \otimes \cdots \otimes I)}{\psi'}
$$

だが、右辺の各項は $\ket{\psi'}$ の左端の量子ビットを測定したときに 0 が得られる確率と 1 が得られる確率の差になっている。
よって、$\ev{Z^{(0)}}{\psi'}$ は $\ket{\psi'}$ の左端の量子ビットをたくさん観測することで推定することができる。
同様の考え方を用いると、$UPU^{\dag}$ の中で $Z$ が作用する量子ビットを測定することで、期待値 $\ev{UPU^{\dag}}{\psi'}$ を推定することができる。

以上の手順を具体例を使って見てみよう。先ほどのノイズなしの VQE で得られた状態を $\ket{\psi}$ とし、ハミルトニアンに含まれる項 $X \otimes X$ の期待値 $\ev{X \otimes X}{\psi}$ を計算してみる。

```python
state = ansatz_two_qubits(res.x)
exp = np.real(state.T.conj() @ np.dot(X[0], X[1]) @ state)
print(exp)  # 厳密な期待値
```

```text
-0.2216815430671909
```

まず、$U = H \otimes H$ を使って $\ket{\psi'} = U\ket{\psi}$ を用意する。$\ket{\psi'}$ のすべての量子ビットを測定したときに、$00, 01, 10, 11$ が得られる確率 $p_{00}, p_{01}, p_{10}, p_{11}$ も計算しておく。

```python
pHad = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
state_XX = np.kron(pHad, pHad) @ state
probs = np.abs(state_XX)**2  # 各測定結果を得る確率
print(probs)
```

```text
[0.19457942 0.30542069 0.30542008 0.19457981]
```

求めたい期待値は $\ev{Z \otimes Z}{\psi'}$ だから、測定結果の確率で書き直すと $p_{00} - p_{01} - p_{10} + p_{11}$ である。
「$\ket{\psi'}$ を $n_\mathrm{shots}$ 回測定して確率を推定し、期待値 $\ev{Z \otimes Z}{\psi'}$ を計算する」という行為を $n_\mathrm{trials}$ 回繰り返して、期待値の推定結果の平均と標準偏差（統計的ゆらぎ）を見てみよう。

```python
ave_list = []
std_list = []
n_shots_list = [10**i for i in range(2, 10)]
n_trials = 10
for n_shots in n_shots_list:
    est_list = []
    for _ in range(n_trials):
        counts = rng.multinomial(n_shots, probs)
        est = (counts[0] - counts[1] - counts[2] + counts[3]) / n_shots
        est_list.append(est)
    ave_list.append(float(np.mean(est_list)))
    std_list.append(float(np.std(est_list)))

print(ave_list)
```

```text
[-0.21600000000000003, -0.2266, -0.22233999999999998, -0.22242,
 -0.22190260000000003, -0.22172155999999998, -0.221678818, -0.221683888]
```

測定回数 $n_\mathrm{shots}$ を大きくとることで、推定値が厳密な期待値に近づいていることがわかる。
この推定値の統計的ゆらぎは $1/\sqrt{n_\mathrm{shots}}$ に比例して小さくなることが知られているので、プロットして確認してみよう。

```python
plt.loglog(n_shots_list, np.abs(ave_list-exp), "o", label="error")
plt.loglog(n_shots_list, std_list, "o", label="std")
plt.loglog(n_shots_list, 1/np.sqrt(n_shots_list), label="1/sqrt(n_shots)")
plt.xlabel("n_shots")
plt.legend()
plt.show()
```

[図 5.12 プレースホルダ: ショット数と推定誤差・標準偏差]

理論的な振る舞い $1/\sqrt{n_\mathrm{shots}}$ にきれいに従っていることがわかる。
実は VQE を用いて高精度な計算を行う場合、この期待値の推定結果の統計的ゆらぎが大きな問題となることがある。
最終的な計算結果の精度を高めるためには、統計的ゆらぎも小さくしなければならないが、これにはかなりの測定回数を必要とする。
量子化学計算において必要な測定回数を見積もった参考文献によると、エネルギー期待値 $E(\theta)$ を 1 回高精度に推定するのにおよそ $1 \times 10^{10}$ 回もの測定が必要になるとされている。
量子コンピュータのクロック（動作速度）がそれほど速くないことを考えると、これはなかなか大変である。